# Classificador Distância Mínima ao Centroide — versão robusta a outliers

*Implementação em numpy · base processada (sem duplicatas, z-score)*

O centroide de cada classe é substituído pela **mediana** de cada atributo, que não é afetada por valores extremos, e a distância euclidiana pela distância de **Manhattan** (L1). A mediana é justamente o ponto que minimiza a soma das distâncias L1 às amostras da classe:

$$\hat{y}(\mathbf{x}) = \arg\min_i \|\mathbf{x} - \tilde{\mathbf{m}}_i\|_1, \qquad \tilde{\mathbf{m}}_i = \text{mediana das amostras da classe } i$$

Protocolo (OBS 1–4 do enunciado): 100 rodadas treino-teste independentes e estratificadas para cada separação 20/80, 30/70, 50/50, 70/30 e 80/20.

## Dados

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"  # antes do numpy: evita falta de memória do OpenBLAS no Windows

import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from avaliacao import (SPLITS, carregar, avaliar, estatisticas, matrizes_confusao, grafico_splits,
                       por_classe, grafico_por_classe, resumo, salvar)

pd.set_option("display.max_rows", 200)

X, y, classes = carregar(processada=True)
print(f"{len(y)} instâncias | {X.shape[1]} atributos | {len(classes)} classes: {list(classes)}")

## Modelo

In [ ]:
class DistanciaMinimaCentroideRobusto:
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.centroides_ = np.array([np.median(X[y == c], axis=0) for c in self.classes_])  # mediana por classe
        return self

    def predict(self, X):
        d = np.abs(X[:, None, :] - self.centroides_[None]).sum(axis=2)  # distância de Manhattan (L1)
        return self.classes_[d.argmin(axis=1)]


MODELOS = {"DMC robusto": DistanciaMinimaCentroideRobusto()}

## Execução
100 rodadas × 5 separações. Em cada rodada, todos os modelos usam a mesma divisão treino/teste.

In [ ]:
res, confusoes = avaliar(MODELOS, X, y, classes)
salvar(res, "classification_processed", "distancia_minima_centroide_robust")

## OBS 1 e 2 — Estatísticas de acurácia e tempos de treinamento/teste (ms)

In [ ]:
estatisticas(res).round(4)

## OBS 2 — Matrizes de confusão da melhor e da pior rodada

In [ ]:
matrizes_confusao(res, confusoes, classes, splits=SPLITS)

## OBS 3 — Efeito da separação treino/teste

In [ ]:
grafico_splits(res)

## OBS 4 — Desempenho por classe (taxa de acerto de cada classe)

In [ ]:
por_classe(res, classes).round(4)

In [ ]:
grafico_por_classe(res, classes)

## Resumo dos resultados

In [ ]:
resumo(res, classes)